# Домашнее задание №17: Полиномиальная регрессия

### 1. Дайте определение полиномиальной регрессии

Полиномиальной регрессией является одна из форм регрессионного анализа, в которой зависимость между независимой переменной х и зависимой переменной у моделируется как п - й степени многочлена в х. Полиномиальная регрессия находит нелинейную зависимость между величиной х и соответствующим условным средним от у.

### 2. Назовите отличительные особенности полиномиальной регрессии

- Моделирует нелинейно разделенные данные (чего не может линейная регрессия). Она более гибкая и может моделировать сложные взаимосвязи.
- Полный контроль над моделированием переменных объекта (выбор степени).
- Необходимо внимательно создавать модель. Необходимо обладать некоторыми знаниями о данных, для выбора наиболее подходящей степени.
- При неправильном выборе степени, данная модель может быть перенасыщена.

### Практическая часть

In [101]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge

import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Собираем данные
data = pd.read_csv('./data/SouthGermanCredit_encoded2.csv')
raw_X = data.iloc[:, :-1]  # все столбцы кроме последнего
y = data.iloc[:, -1]   # целевая переменная "Кредитный риск" (0:плохой, 1:хороший)

# Генерируем матрицу всех полиномных комбинаций
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_X = poly.fit_transform(raw_X)          # новые столбцы: квадраты и произведения

# Нормализуем
scaler = MinMaxScaler()
X = scaler.fit_transform(poly_X)

In [102]:
# Обучаем на линейной регрессии
model = Ridge().fit(X, y)


Посмотрим долю тех кого обучение предсказало верно

In [103]:
y_pred = model.predict(X)
y_class = (y_pred > 0.5).astype(int)
print('Точность:',(y_class == y).mean()*100,'%')

Точность: 99.19839679358718 %


Матрица ошибок

In [104]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y, y_class))

[[291   8]
 [  0 699]]


Какая-то магия конечно, на степени полинома 2, было 99%, а на 3 стало 100%

Что нам покажем родной скор

In [105]:
print(model.score(X, y))

0.9096436943493168


Сделал копию файла с выборкой. Уберу оттуда записи 23 (плохая) и 24 (хорошая).

Теперь проверюсь на них. Он на них не мог обучиться.

In [106]:
data_full = pd.read_csv('./data/SouthGermanCredit_encoded.csv')
raw_X_full = data_full.iloc[:, :-1]
y_full = data_full.iloc[:, -1]

held = raw_X_full.iloc[[22, 23]]                   # сырые строки, таблицей
X_held = scaler.transform(poly.transform(held))    # каждый transform ровно один раз

pred = model.predict(X_held)
print('предсказание:', pred)
print('классы:     ', (pred > 0.5).astype(int))
print('факт:       ', y_full.iloc[[22, 23]].values)

предсказание: [-0.16147261  0.25299231]
классы:      [0 0]
факт:        [0 1]


In [107]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, accuracy_score
from sklearn.pipeline import make_pipeline

pipe = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    MinMaxScaler(),
    Ridge()
)

def acc(y_true, y_pred):
    return accuracy_score(y_true, (y_pred > 0.5).astype(int))

accs = cross_val_score(pipe, raw_X, y, cv=5, scoring=make_scorer(acc))
print(accs.mean())     # средняя accuracy

0.5909698492462312


Я вручную прятал 2 строки для обучения. Потом почитал как это делают профи.

Есть `cross_val_score`, как она работает:
- Разбивает данные на "фолды", в данном случае на 5.
- Берёт фолд 1 как тест: обучает модель на остальных четырёх, предсказывает фолд 1, считает score.
- Повторяет для фолдов 2–5: каждый раз заново обучается на четырёх и проверяет на пятом.

Результат 59% для полиномов второй степени. А без полиномов получается 62%.

Идея получилась снова лучше, чем реализация, по крайней мере для этой выборки.
